# 1.06 — Lists — solutions

Lesson: [`lessons/1.06_lists.md`](../lessons/1.06_lists.md)

Unless a problem says "in place", **don't modify the input list** — build and return a new one.

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — mean and median

You know the maths — now write it. No `statistics` module. `median` of an even-length list is the mean of the two middle values. Both return `float`.

In [ ]:
def mean(nums):
    return sum(nums) / len(nums)

def median(nums):
    s = sorted(nums)
    n = len(s)
    mid = n // 2
    if n % 2 == 1:
        return float(s[mid])
    return (s[mid - 1] + s[mid]) / 2

In [ ]:
check(mean, [(([1, 2, 3, 4],), 2.5), (([10],), 10.0)])
check(median, [(([3, 1, 2],), 2.0), (([4, 1, 3, 2],), 2.5), (([7],), 7.0)])

## Problem 2 — remove duplicates, keep order

`remove_duplicates(items)` keeps the first occurrence of each item, in the original order.

In [ ]:
def remove_duplicates(items):
    seen = []
    for x in items:
        if x not in seen:
            seen.append(x)
    return seen

In [ ]:
check(remove_duplicates, [(([3, 1, 3, 2, 1],), [3, 1, 2]), ((["a", "a"],), ["a"]), (([],), [])])

## Problem 3 — second largest, in one pass

`second_largest(nums)` returns the second-largest **distinct** value, or `None` if there isn't one. Walk the list once — don't sort. Plan: which two variables do you need to keep updated?

In [ ]:
def second_largest(nums):
    first = second = None
    for x in nums:
        if first is None or x > first:
            second = first
            first = x
        elif x != first and (second is None or x > second):
            second = x
    return second

In [ ]:
check(second_largest, [(([5, 1, 9, 7],), 7), (([9, 9, 8],), 8), (([4, 4],), None), (([2],), None), (([-3, -1, -2],), -2)])

## Problem 4 — rotate

`rotate(items, k)` rotates right by `k` places: `rotate([1, 2, 3, 4, 5], 2)` → `[4, 5, 1, 2, 3]`. `k` may be larger than the length. Hint: slicing and `%`.

In [ ]:
def rotate(items, k):
    if not items:
        return []
    k = k % len(items)
    return items[-k:] + items[:-k] if k else items[:]

In [ ]:
check(rotate, [(([1, 2, 3, 4, 5], 2), [4, 5, 1, 2, 3]), (([1, 2, 3], 3), [1, 2, 3]), (([1, 2, 3], 4), [3, 1, 2]), (([], 5), [])])

## Problem 5 — chunks

`chunk(items, size)` → `chunk([1, 2, 3, 4, 5], 2)` = `[[1, 2], [3, 4], [5]]`.

In [ ]:
def chunk(items, size):
    out = []
    for start in range(0, len(items), size):
        out.append(items[start:start + size])
    return out

In [ ]:
check(chunk, [(([1, 2, 3, 4, 5], 2), [[1, 2], [3, 4], [5]]), (([1, 2, 3], 5), [[1, 2, 3]]), (([], 3), [])])

## Problem 6 — running total

`running_total([100, 50, 25])` → `[100, 150, 175]`.

In [ ]:
def running_total(nums):
    out = []
    total = 0
    for x in nums:
        total += x
        out.append(total)
    return out

In [ ]:
check(running_total, [(([100, 50, 25],), [100, 150, 175]), (([],), []), (([5, -5, 5],), [5, 0, 5])])

## Problem 7 — transpose

`transpose(matrix)` swaps rows and columns: `[[1, 2, 3], [4, 5, 6]]` → `[[1, 4], [2, 5], [3, 6]]`. Use nested loops.

In [ ]:
def transpose(matrix):
    if not matrix:
        return []
    rows, cols = len(matrix), len(matrix[0])
    out = []
    for c in range(cols):
        new_row = []
        for r in range(rows):
            new_row.append(matrix[r][c])
        out.append(new_row)
    return out

In [ ]:
check(transpose, [(([[1, 2, 3], [4, 5, 6]],), [[1, 4], [2, 5], [3, 6]]), (([[7]],), [[7]]), (([],), [])])

## Problem 8 — month-over-month change

`pct_changes(values)` returns the percentage change between each consecutive pair, rounded to 1 decimal: `[100, 110, 99]` → `[10.0, -10.0]`.

In [ ]:
def pct_changes(values):
    out = []
    for i in range(1, len(values)):
        change = (values[i] - values[i - 1]) / values[i - 1] * 100
        out.append(round(change, 1))
    return out

In [ ]:
check(pct_changes, [(([100, 110, 99],), [10.0, -10.0]), (([50],), []), (([200, 250, 250],), [25.0, 0.0])])

## Problem 9 ★ — moving average

`moving_average(nums, window)` returns the average of each full window, rounded to 2 decimals: `moving_average([1, 2, 3, 4, 5], 3)` → `[2.0, 3.0, 4.0]`.

In [ ]:
def moving_average(nums, window):
    out = []
    for i in range(len(nums) - window + 1):
        out.append(round(sum(nums[i:i + window]) / window, 2))
    return out

In [ ]:
check(moving_average, [(([1, 2, 3, 4, 5], 3), [2.0, 3.0, 4.0]), (([10, 20], 3), []), (([1, 2, 4], 2), [1.5, 3.0])])

## Problem 10 ★ — merge two sorted lists

`merge_sorted(a, b)` merges two already-sorted lists into one sorted list **without** calling `sorted()` or `.sort()`. Plan: one pointer into each list; repeatedly take the smaller front item.

In [ ]:
def merge_sorted(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i])
            i += 1
        else:
            out.append(b[j])
            j += 1
    out.extend(a[i:])
    out.extend(b[j:])
    return out

In [ ]:
check(merge_sorted, [(([1, 4, 9], [2, 3, 10, 11]), [1, 2, 3, 4, 9, 10, 11]), (([], [1]), [1]), (([2, 2], [2]), [2, 2, 2])])